In [1]:
import fitz  # PyMuPDF
from pathlib import Path

# Function to extract text from PDF
def extract_pdf_text(pdf_path):
    file = Path(pdf_path)
    if not file.exists():
        raise FileExistsError
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text")  # Extract text
    return text


In [2]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to generate embeddings from text
def get_text_embeddings(text):
    return model.encode(text)


/Users/neerajmokha/Documents/ajangid_Github/ml_projects/faiss_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import faiss
import numpy as np

# Initialize FAISS index for cosine similarity (L2 distance)
def initialize_faiss_index(dim=384):
    index = faiss.IndexFlatL2(dim)  # Using L2 distance (Euclidean) for cosine similarity
    return index

# Function to store text embeddings in FAISS
def store_embeddings_in_faiss(text_chunks):
    index = initialize_faiss_index(dim=384)  # Embedding dimension for 'all-MiniLM-L6-v2'
    embeddings = [get_text_embeddings(chunk) for chunk in text_chunks]
    embeddings = np.array(embeddings).astype('float32')

    # Add embeddings to FAISS index
    index.add(embeddings)
    return index


In [4]:
def retrieve_context_from_faiss(query, index, text_chunks, k=3):
    query_embedding = get_text_embeddings(query).reshape(1, -1).astype('float32')
    
    # Perform the similarity search (retrieve top k most similar documents)
    _, indices = index.search(query_embedding, k)  # k = number of top results
    
    # Retrieve the corresponding text chunks
    return [text_chunks[i] for i in indices[0]]


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the TinyLlama model (replace 'tiny-llama' with your local TinyLlama model name)
model_name = "tinyllama"  # Example: Use the actual model path
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Function to generate a response from TinyLlama
def query_tiny_llama(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)
    output = model.generate(inputs['input_ids'], max_length=150, num_return_sequences=1)
    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    return answer

# Function to generate an answer based on context
def generate_answer_with_context(query, index, text_chunks):
    # Step 1: Retrieve the most relevant context from FAISS
    retrieved_context = retrieve_context_from_faiss(query, index, text_chunks)

    # Step 2: Combine the retrieved context with the query
    context = "\n".join(retrieved_context)
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"

    # Step 3: Generate the answer using TinyLlama
    answer = query_tiny_llama(prompt)
    return answer


OSError: tinyllama is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
import requests
import ollama

# Function to query Ollama with the prompt
def query_ollama(prompt):
    # response = ollama.chat(model='tinyllama', messages=[
    #     {
    #         'role': 'user',
    #         'content': prompt
    #     },
    # ])
    url = "http://localhost:11434/api/generate"  # Adjust with the correct model name
    headers = {"Content-Type": "application/json"}
    data = {"prompt": prompt, "max_tokens": 150, "model": "tinyllama", "stram": False}
    
    response = requests.post(url, headers=headers, json=data)
    
    return response

# Function to generate the answer with retrieved context
def generate_answer_with_context(query, index, text_chunks):
    # Step 1: Retrieve the most relevant context from FAISS
    retrieved_context = retrieve_context_from_faiss(query, index, text_chunks)

    # Step 2: Combine the retrieved context and the user query into the prompt
    context = "\n".join(retrieved_context)
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"

    # Step 3: Generate the answer using Ollama
    answer = query_ollama(prompt)
    
    return answer


In [7]:
# Example Workflow: Load PDF, extract text, generate embeddings, and answer a query

# 1. Extract text from PDF
pdf_text = extract_pdf_text("/Users/neerajmokha/Desktop/Anu Interview/AnuradhaJangid_Resume.pdf")  # Specify your PDF file path
text_chunks = pdf_text.split("\n")  # Split text into chunks (e.g., paragraphs)



In [8]:
# 2. Store embeddings in FAISS
index = store_embeddings_in_faiss(text_chunks)



In [14]:
# 3. Query input from user
import ollama
user_query = "who is this document about and what is theri name?"

# 4. Generate an answer based on retrieved context
answer = generate_answer_with_context(user_query, index, text_chunks)

print("Generated Answer:", answer)


AttributeError: 'Response' object has no attribute 'model_dump_json'